# Construir el dataset combinado del equipo

Junta cuatro datasets públicos de Kaggle (Fruits-360, Fresh and Rotten, PackEat y Vegetable Image Dataset) en `train/` y `test/`. Las fotos del equipo van aparte, en una carpeta de Google Drive, y las suma el notebook de entrenamiento.

**No hace falta Add Input:** el notebook descarga cada dataset con `kagglehub`. Solo activa **Settings → Internet: On**. No hace falta GPU.

Al terminar, el dataset queda en `/kaggle/working/dataset` con `train/`, `test/`, `manifest.csv`, `conteos.csv` y `reporte.json`.

In [ ]:
import os, subprocess

REPO = "/kaggle/working/repo"
if not os.path.isdir(REPO):
    r = subprocess.run(["git", "clone", "https://github.com/11909/Clasificador-fruta-verdura.git", REPO],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("No se pudo clonar el repo. Activa Settings → Internet en el panel derecho "
                           "(Kaggle pide verificar tu teléfono) y vuelve a correr esta celda.\n" + r.stderr)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"])  # trae la última versión de main
%cd {REPO}
!pip install -q -r requirements.txt

## 1. Descargar las fuentes y revisar su estructura

Descarga cada dataset y muestra sus carpetas. Si un split no coincide con `configs/dataset.yaml`, corrígelo en la celda siguiente.

In [ ]:
import yaml
from src.build_dataset import explorar, preparar_fuentes

cfg = preparar_fuentes(yaml.safe_load(open("configs/dataset.yaml")))
for f in cfg["fuentes"]:
    print(f"\n=== {f['nombre']} ===")
    raiz = f.get("raiz") or f.get("ruta") or next(iter(f["splits"].values()))
    explorar(raiz[0] if isinstance(raiz, list) else raiz, profundidad=4)

## 2. Ajustar la config (si hace falta)

In [ ]:
# Ejemplos de ajuste (las rutas ya son absolutas):
# fuente = next(f for f in cfg["fuentes"] if f["nombre"] == "freshrotten")
# fuente["splits"]["train"] = fuente["raiz"] + "/otra/ruta/Train"
# fuente["nivel_clase"] = 1
# cfg["fuentes"] = [f for f in cfg["fuentes"] if f["nombre"] != "packeat"]   # quitar una fuente
for f in cfg["fuentes"]:
    print(f["nombre"], f.get("splits", f.get("ruta")), "nivel_clase =", f.get("nivel_clase", 0))

## 3. Construir

In [ ]:
import json
from src.build_dataset import construir

reporte = construir(cfg)
print(json.dumps(reporte, indent=2, ensure_ascii=False))

## Desbalance de clases

Imágenes por etiqueta y las clases de cada etiqueta, de más a menos.

In [ ]:
import pandas as pd

conteos = pd.read_csv(f"{cfg['salida']}/conteos.csv")
conteos["total"] = conteos["train"] + conteos["test"]
display(conteos.groupby("etiqueta")[["train", "test", "total"]].sum())
por_clase = conteos.pivot_table(index=["etiqueta", "clase"], columns="fuente", values="total", aggfunc="sum", fill_value=0)
por_clase["total"] = por_clase.sum(axis=1)
display(por_clase.sort_values(["etiqueta", "total"], ascending=[True, False]))

## 4. Publicarlo como dataset del equipo

1. **Save Version → Save & Run All** para guardar la salida.
2. La primera vez: en la versión guardada, pestaña **Output** → **New Dataset**. Déjalo **privado** y compártelo con el equipo (**Settings → Sharing**).
3. Las siguientes veces (cambió el mapeo o las fuentes): corre de nuevo y usa **New Version** sobre el mismo dataset.

Este dataset solo se actualiza volviendo a correr el notebook. Las fotos del equipo van en la carpeta de Google Drive.